In [0]:
%sql
CREATE VOLUME IF NOT EXISTS dbx_proj4.default.project4_checkpoints;

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Project 4 - Databricks Streaming Bronze Layer (Event Hub -> Delta)
# MAGIC
# MAGIC Structured Streaming read from Event Hub `eh-telematics` (consumer group
# MAGIC `databricks-cg` - Synapse reads the same hub independently via its own
# MAGIC `synapse-cg`, so neither platform's checkpoint/offset tracking interferes with the
# MAGIC other's). Writes straight to a bronze Delta table, matching the medallion pattern
# MAGIC already used for the batch leg (raw landing, minimal transform), so silver/gold
# MAGIC logic can be reused downstream the same way it is for batch.
# MAGIC
# MAGIC **Trigger mode: `availableNow=True`, not continuous.** This processes everything
# MAGIC currently sitting in the Event Hub and then stops - not an always-on streaming job.
# MAGIC Deliberate cost choice against the $127 budget, not a technical limitation. A
# MAGIC production system doing genuine real-time ingestion would use a continuous or
# MAGIC fixed-interval trigger instead; that tradeoff is itself worth a line in the
# MAGIC platform comparison doc.
# MAGIC
# MAGIC **One-time setup before running:**
# MAGIC 1. Cluster needs the Maven library
# MAGIC    `com.microsoft.azure:azure-eventhubs-spark_2.12:2.3.22` (or the latest 2.3.x
# MAGIC    release) attached - Cluster > Libraries > Install New > Maven. This is the
# MAGIC    Spark <-> Event Hub connector, a different package from the plain
# MAGIC    `azure-eventhub` SDK the producer notebook uses.
# MAGIC 2. Run `nb_streaming_producer_databricks` first - nothing to consume otherwise.
# MAGIC 3. Reads the same `project4-eventhub` secret scope / `eventhub-connection-string`
# MAGIC    secret as the producer. One connection string works for both send and listen
# MAGIC    here since it's the same Key Vault secret - in a real deployment you'd scope
# MAGIC    producer and consumer to separate SAS policies (Send-only vs Listen-only)
# MAGIC    rather than share one. Worth calling out as a shortcut taken for project speed,
# MAGIC    not best practice.
# MAGIC 4. `CHECKPOINT_PATH` below needs to point somewhere this workspace can actually
# MAGIC    write durable state to (a mounted ADLS path or Unity Catalog volume) - adjust
# MAGIC    it to match your environment before running.

In [0]:
SECRET_SCOPE = "project4-eventhub"
CONNECTION_STRING_SECRET = "eventhub-connection-string"
EVENTHUB_NAMESPACE = "evhns-project4"
EVENT_HUB_NAME = "eh-telematics"
CONSUMER_GROUP = "databricks-cg"
CHECKPOINT_PATH = "/Volumes/dbx_proj4/default/project4_checkpoints/streaming_bronze_telematics"

In [0]:
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, IntegerType
 
 
def log_pipeline_run(spark, platform, layer, start_dt, end_dt):
    duration = round((end_dt - start_dt).total_seconds(), 2)
    log_row = spark.createDataFrame([{
        "platform": platform, "layer": layer,
        "start_ts": start_dt, "end_ts": end_dt, "duration_seconds": duration,
    }])
    log_row.write.format("delta").mode("append").saveAsTable("pipeline_run_log")
    print(f"[{platform}/{layer}] duration: {duration}s")

In [0]:
# MAGIC %md
# MAGIC ## Build Event Hub connection config
# MAGIC The Spark Event Hub connector needs its own encrypted connection-string wrapper
# MAGIC (`EventHubsUtils.encrypt`), not the raw string - a different requirement from the
# MAGIC plain SDK the producer notebook uses. No direct PySpark wrapper for this utility
# MAGIC exists, so it's called through Spark's JVM gateway.

In [0]:
connection_str = dbutils.secrets.get(scope=SECRET_SCOPE, key=CONNECTION_STRING_SECRET)
 
kafka_bootstrap_servers = f"{EVENTHUB_NAMESPACE}.servicebus.windows.net:9093"
sasl_jaas_config = (
    'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
    f'username="$ConnectionString" password="{connection_str}";'
)

In [0]:
# MAGIC %md
# MAGIC ## Read stream, parse payload, write to bronze
# MAGIC Event Hub delivers the message body as `binary` in the `body` column, plus its own
# MAGIC envelope metadata (`enqueuedTime`, `offset`, `sequenceNumber`, partition info) -
# MAGIC kept alongside the parsed payload rather than discarded, since offset/sequence
# MAGIC number is useful for debugging exactly-once/at-least-once behavior later.

In [0]:
event_schema = StructType([
    StructField("device_id", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("PID", StringType(), True),
    StructField("value", DoubleType(), True),
    StructField("alarm_class", IntegerType(), True),
])
 
raw_stream = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", kafka_bootstrap_servers)
    .option("subscribe", EVENT_HUB_NAME)
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.jaas.config", sasl_jaas_config)
    .option("kafka.group.id", CONSUMER_GROUP)
    .option("kafka.request.timeout.ms", "60000")
    .option("kafka.session.timeout.ms", "30000")
    .option("startingOffsets", "earliest")
    .load()
    .withColumnRenamed("timestamp", "kafka_message_timestamp")
    .withColumnRenamed("offset", "eh_offset")
    .withColumnRenamed("partition", "eh_partition")
)
 
parsed_stream = (
    raw_stream
    .withColumn("payload", F.col("value").cast("string"))
    .withColumn("parsed", F.from_json(F.col("payload"), event_schema))
    .select(
        "parsed.device_id", "parsed.timestamp", "parsed.PID", "parsed.value", "parsed.alarm_class",
        "kafka_message_timestamp", "eh_offset", "eh_partition",
    )
    .withColumn("ingestion_timestamp", F.current_timestamp())
)

In [0]:
start_dt = datetime.now(timezone.utc)
 
streaming_query = (
    parsed_stream.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", CHECKPOINT_PATH)
    .toTable("bronze_telematics_stream")
)
 
streaming_query.awaitTermination()  # blocks until availableNow finishes draining the hub
 
end_dt = datetime.now(timezone.utc)
log_pipeline_run(spark, "Databricks", "streaming_bronze", start_dt, end_dt)

Sanity check

In [0]:
bronze_stream_check = spark.read.table("bronze_telematics_stream")
 
print("bronze_telematics_stream rows:", bronze_stream_check.count())
print("null device_id rows (should be 0 - a parse failure):",
      bronze_stream_check.filter(F.col("device_id").isNull()).count())
print("min/max ingestion_timestamp:")
bronze_stream_check.select(F.min("ingestion_timestamp"), F.max("ingestion_timestamp")).show(truncate=False)